Deep Learning Embeddings Roadmap: Co-occurrence ->
Word2Vec -> GloVe
Phase 1: Co-occurrence Matrix
Goal: Understand the raw statistics behind word embeddings.

1. Build a small corpus manually (10–20 sentences).
2. Construct the co-occurrence matrix X, where X_ij = count of word j in context of word
   i.
3. Normalize rows to get probabilities P(j | i).
4. Observe that words appearing in similar contexts have similar row vectors.
   Phase 2: Word2Vec (Skip-gram / CBOW)
   Goal: See how predictive models turn raw counts into dense embeddings.
5. Implement skip-gram on your small corpus.
6. Optional: Add negative sampling to approximate softmax.
7. Train for a few epochs and extract learned embeddings.
8. Compute cosine similarity between words.
9. Compare embeddings with raw co-occurrence vectors to see compression into lower
   dimensions.
   Phase 3: Analogy and Similarity Tasks
   Goal: See emergent semantics from embeddings.
10. Test word relationships: e.g., "king - man + woman".
11. Observe how skip-gram captures semantic relationships unlike raw co-occurrence
    matrices.
    Phase 4: GloVe (Global Vectors)
    Goal: Bridge count-based and predictive embeddings.
12. Understand GloVe objective: J = sum_ij f(X_ij) (w_i^T w_j + b_i + b_j - log X_ij)^2.
13. Use your co-occurrence matrix from Phase 1.
14. Implement simple matrix factorization (SVD) on log(X).
15. Compare embeddings to skip-gram embeddings.
16. Observe that both capture semantic similarity via different mechanisms (predictive
    vs count-based).
    Phase 5: Visualization & Intuition Check
17. Reduce embeddings to 2D using PCA/t-SNE.
18. Plot words like "cat", "dog", "king", "queen".
19. Compare raw co-occurrence vectors, skip-gram embeddings, and GloVe embeddings.
20. Insight: embeddings encode context similarity, which produces semantics.
    Outcome After These Experiments

- Understand math behind embeddings (raw counts -> predictive -> global).
- See embeddings emerge visually.
- Understand why Word2Vec and GloVe work and how they relate.
- Ready for contextual embeddings (ELMo, BERT) and transformers with full
  understanding.


way to do the nlp


- do the cooccurance matrix -- with 1 if present if not 0 ###not gona do it due to it is not feasible practically

- the co occurance matrix with the PMI just construct with small sample

  - two ways if less then zero then zero
  - positive pmi

- dimension reduction using svd


In [2]:
import numpy as np
import nltk
from scipy.sparse.linalg import svds
import sys
nltk.download('brown')

from nltk.corpus import brown



[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\sp710\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [3]:
# constants

WINDOW_SIZE: int=3

k:int= 50

In [4]:
sentences=brown.sents()[:100]

corpus = [sentence[i:i+WINDOW_SIZE] for sentence in sentences for i in range(len(sentence)-WINDOW_SIZE+1)]

print(corpus)

[['The', 'Fulton', 'County'], ['Fulton', 'County', 'Grand'], ['County', 'Grand', 'Jury'], ['Grand', 'Jury', 'said'], ['Jury', 'said', 'Friday'], ['said', 'Friday', 'an'], ['Friday', 'an', 'investigation'], ['an', 'investigation', 'of'], ['investigation', 'of', "Atlanta's"], ['of', "Atlanta's", 'recent'], ["Atlanta's", 'recent', 'primary'], ['recent', 'primary', 'election'], ['primary', 'election', 'produced'], ['election', 'produced', '``'], ['produced', '``', 'no'], ['``', 'no', 'evidence'], ['no', 'evidence', "''"], ['evidence', "''", 'that'], ["''", 'that', 'any'], ['that', 'any', 'irregularities'], ['any', 'irregularities', 'took'], ['irregularities', 'took', 'place'], ['took', 'place', '.'], ['The', 'jury', 'further'], ['jury', 'further', 'said'], ['further', 'said', 'in'], ['said', 'in', 'term-end'], ['in', 'term-end', 'presentments'], ['term-end', 'presentments', 'that'], ['presentments', 'that', 'the'], ['that', 'the', 'City'], ['the', 'City', 'Executive'], ['City', 'Executive'

In [5]:
def PPMI(cooccurance,N):
    '''
        PMI(w, c) = log p(c|w)
        p(c)
        = log |  count(w, c) ∗ N    |
              |-------------------  |
              |count(c) ∗ count(w)  |
    '''
    countWords = np.sum(cooccurance, axis=1)
    
    print(countWords)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        expected = np.outer(countWords, countWords) / N
        pmi = np.log10(cooccurance / expected)    
        pmi[np.isnan(pmi)] = 0
        pmi[pmi < 0] = 0
        
    cooccurance[:, :] = pmi
    return cooccurance



In [ ]:

words=sorted(set(word for sentence in sentences for word in sentence))

word2id = {w: i for i, w in enumerate(words)}
id2word = {i: w for w, i in word2id.items()}

#calculating the coocurance matrix

cooccurance=np.zeros((len(words),len(words)),dtype=float)

for window in corpus:
    for i, word in enumerate(window):
        w_idx = word2id[word]

        for j, neighbor in enumerate(window):
            if i != j:
                n_idx = word2id[neighbor]
                cooccurance[w_idx, n_idx] += 1
np.set_printoptions(threshold=sys.maxsize)

print("Co-occurrence sum:", len(corpus),np.sum(cooccurance))

Co-occurrence sum: 2069 12414.0


In [7]:
# calculating the ppmi mat
ppmi_mat=PPMI(cooccurance,len(words))

print(ppmi_mat)


[  6.   6.   6.   6.   6.   6. 174.  14.  10. 516.  22. 178.  16.   6.
   6.   6.   6.   4.   4.   4.   6.   6.   6.  10.   4.   4.   6.   4.
   6.   6.   4.  10.   8.   4.   4.   4.  12.   6.   2.   2.   6.  30.
  10.   2.   6.   2.  12.  12.   6.  14.   2.   4.   6.   6.  12.   6.
   6.  18.   2.   4.   6.  10.   2.   6.   4.   6.   6.  16.   6.   6.
   6.  14.   6.   6.   0.  58.  12.   6.   6.  14.   6.  46.   6.   2.
   6.   2.   6.   6.   6.   4.  12.   4.  10.   2.   2.  22.  78.   6.
   6.   6.   6.  18.  24.  20.   6.   6.   8.   6.  24.   4.   6.   2.
  20.   4.   6.  30.   2.  14.   2.   2.  12.  12.   6.   6.   6.   6.
  12.   6.  16.   6.   6.   4.   6.  24.   6.   6.   4.   2.   6.   6.
   8.   6.   2.   0.  12.  28.   6.   2.   6.   6.   2.   4.   6.   4.
   6.   4.   6.  22.   6.   6.   6.   6.   6.   6.   2.   8.  26.  12.
  12.   6.   2.  12.   2.   6.   6.   2.   6.  12.  12.   4.   2.   4.
  32.  12.  12.   6.   6.  14.  60.   4.   4.   2.   8.   6.   6.   6.
  12. 

In [8]:
### doing the svd (singular value decomposition on it)

def svd(matrix):
    
    U, S, Vt = svds(matrix,k=k)

    return [U[:, ::-1], S[::-1], Vt[::-1, :]]
    

svd_ppmi=svd(ppmi_mat)

for i in svd_ppmi:
    print(np.shape(i))
    
### and then doing the approximation

Word_embeding = svd_ppmi[0] @ np.diag(svd_ppmi[1])

print(f"the size of the new Embeding is {np.shape(Word_embeding)}")

(860, 50)
(50,)
(50, 860)
the size of the new Embeding is (860, 50)


In [9]:
# print(Word_embeding)

print(words)
print(np.dot(Word_embeding[word2id['Fulton']],Word_embeding[word2id['County']].T))

['$10', '$100', '$3', '$30', '$4', '$50', "''", '(', ')', ',', '--', '.', '1', '1,119', '13', '13th', '18', '1913', '1923', '1937', '1958', '1961', '1962', '2', '29-5', '402', '637', '71', '74', '8', '87-31', ':', 'A', 'After', 'Aj', 'Ala.', 'Allen', 'Alpharetta', 'As', 'Ask', 'Association', 'Atlanta', "Atlanta's", 'Attorneys', 'Aug.', 'Austin', 'Authority', 'B.', 'Bar', 'Barber', 'Before', 'Being', 'Bellwood', 'Berry', 'Blue', 'Board', 'Bowden', 'Bush', 'But', 'Byrd', "Byrd's", 'Caldwell', "Caldwell's", 'Callan', 'Carey', 'Chairman', 'Cheshire', 'City', 'Colquitt', 'Commerce', "Commissioner's", 'Committee', 'Congress', 'Constitution', 'Construction', 'County', 'Court', 'D.', "Daniel's", 'Davis', 'Democratic', 'Department', "Department's", 'Despite', 'Dorsey', 'During', 'Durwood', 'E.', 'Education', 'Everything', 'Executive', 'Failure', 'Felix', 'Five', 'Four', 'Friday', 'Fulton', 'GOP', 'Gainesville', 'Garland', 'George', 'Georgia', "Georgia's", 'Gov.', 'Grady', 'Grand', 'Griffin', 'H

# Continious Bag of word

In [10]:
# datasets
np_corpus = np.array(corpus)
print(np.shape(np_corpus))

X_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]), 0 : WINDOW_SIZE-1]
Y_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]) , WINDOW_SIZE-1:WINDOW_SIZE]

X_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , 0 : WINDOW_SIZE-1]
Y_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , WINDOW_SIZE-1:WINDOW_SIZE]


print(np.shape(X_train),np.shape(Y_train))
print(np.shape(X_test),np.shape(Y_test))


(2069, 3)
(1448, 2) (1448, 1)
(621, 2) (621, 1)


In [20]:
# constants

L=2 #is the no if layer

MAX_ITR=100 

LEARNING_RATE=0.5 #η 

NO_OF_INPUT=[(WINDOW_SIZE-1)*len(words) , (WINDOW_SIZE-1) * k , len(words)] # mem friendly input h1,h2 output

NO_OF_OUTPUT=len(words)

BATCH_SIZE=10

In [ ]:

def initialize_parameters(layer_dims, method="xavier"):
    """
    Initializes weights and biases for a fully connected neural network.
    
    Parameters:
    -----------
    layer_dims : list of int
        Sizes of each layer in the network. Example: [784, 128, 64, 10]
    method : str
        Initialization method: "xavier" or "he"
    
    Returns:
    --------
    weights : list of np.ndarray
        Weight matrices for each layer
    biases : list of np.ndarray
        Bias vectors for each layer
    """
    weights = [] #(784,128) , (128,64) ,(64,10)
    biases = [] #(764 x 1 , 128 x 1 , 64 x 1 , 10 x 1)
    
    for i in range(len(layer_dims)-1):
        n_in = layer_dims[i]
        n_out = layer_dims[i+1]
        
        if method == "xavier":
            W = np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in)
        elif method == "he":
            W = np.random.randn(n_in,n_out) * np.sqrt(2.0 / n_in)
        else:
            raise ValueError("Invalid method. Use 'xavier' or 'he'.")
        
        b = np.zeros((1, n_out))
        
        weights.append(W)
        biases.append(b)
    
    return weights, biases

weight,bias=initialize_parameters(NO_OF_INPUT)

print(weight[0].shape)
print(weight[1].shape)
print(bias[0].shape)
print(bias[1].shape)




(1720, 100)
(100, 860)
(1, 100)
(1, 860)


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def loss_fn(y_pred, y_true):
    """
    Cross-entropy loss
    y_pred: (batch_size, vocab_size)
    y_true: (batch_size, vocab_size) one-hot
    """
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-9)) / m

def eigen(output, word2id, vocab_size):
    """
    Convert words to one-hot vectors
    output: list/array of words (batch_size,)
    """
    idxs = [word2id[w] for w in output]
    ans = np.eye(vocab_size)[idxs]
    return ans

def update_parameters(grads_w, grads_b):
    for i in range(L):
        weight[i] -= LEARNING_RATE * grads_w[i]
        bias[i]   -= LEARNING_RATE * grads_b[i]

def forwardpropogation(x): 
    """
    Forward pass
    x: (batch_size, input_dim)
    """
    activation, preactivation = [x], []

    for i in range(L):
        z = np.dot(activation[-1], weight[i]) + bias[i]
        preactivation.append(z)

        if i == L - 1:
            a = softmax(z)
        else:
            a = sigmoid(z)

        activation.append(a)

    return activation, preactivation

def backpropogation(activation, preactivation, y):
    """
    Backward pass
    y: one-hot (batch_size, vocab_size)
    """
    m = y.shape[0]
    grad_w = [None] * L
    grad_b = [None] * L

    # output layer error
    dz = activation[-1] - y  

    for i in reversed(range(L)):
        A_prev = activation[i]

        grad_w[i] = np.dot(A_prev.T, dz) / m
        grad_b[i] = np.sum(dz, axis=0, keepdims=True) / m

        if i > 0:
            dA_prev = np.dot(dz, weight[i].T)
            sig = sigmoid(preactivation[i-1])
            dz = dA_prev * sig * (1 - sig)

    return grad_w, grad_b

def windows_to_concat_onehot(batch_windows, word2id, vocab_size):
    """
    batch_windows: (batch_size, WINDOW_SIZE-1) of words
    returns: (batch_size, (WINDOW_SIZE-1)*vocab_size) concatenated one-hot
    """
    batch_size, context_len = batch_windows.shape
    idxs = np.vectorize(word2id.get)(batch_windows)   # convert words -> indices
    onehots = np.eye(vocab_size)[idxs]                # (batch, context_len, vocab_size)
    return onehots.reshape(batch_size, -1)      


In [21]:
######################################################## MINI_BATCH_GD################################ 
for epoch in range(MAX_ITR):
    # shuffle
    idx = np.arange(len(X_train))
    np.random.shuffle(idx)
    X_train = X_train[idx]
    Y_train = Y_train[idx]

    print(f"\nEpoch {epoch+1}/{MAX_ITR}")
    print("Sample batch:", X_train[:2], Y_train[:2])

    for i in range(0, len(X_train), BATCH_SIZE):
        # --------- prepare batch inputs ---------
        X_batch_words = X_train[i : i + BATCH_SIZE]               # (batch, WINDOW_SIZE-1)
        Y_batch_words = Y_train[i : i + BATCH_SIZE].reshape(-1)   # (batch,)

        # context → concat one-hot
        X_batch = windows_to_concat_onehot(X_batch_words, word2id, len(word2id))

        # labels → one-hot
        Y_batch = eigen(Y_batch_words, word2id, len(word2id))     # (batch, vocab_size)

        # --------- forward pass ---------
        activation, preactivation = forwardpropogation(X_batch)

        # --------- compute loss ---------
        loss = loss_fn(activation[-1], Y_batch)
        if i % (BATCH_SIZE * 10) == 0:   # print every 10 batches
            print(f"  batch {i//BATCH_SIZE:03d}: loss={loss:.4f}")

        # --------- backward + update ---------
        dW, db = backpropogation(activation, preactivation, Y_batch)
        update_parameters(dW, db)



Epoch 1/100
Sample batch: [['operated' 'in']
 ['work' 'with']] [['a']
 ['city']]
  batch 000: loss=7.0171
  batch 010: loss=5.9707
  batch 020: loss=6.8053
  batch 030: loss=6.7194
  batch 040: loss=5.8030
  batch 050: loss=5.9996
  batch 060: loss=5.2580
  batch 070: loss=6.2151
  batch 080: loss=6.2544
  batch 090: loss=6.5631
  batch 100: loss=6.2140
  batch 110: loss=6.6630
  batch 120: loss=5.8217
  batch 130: loss=6.5034
  batch 140: loss=5.8962

Epoch 2/100
Sample batch: [['the' 'City']
 ['elected' 'servants']] [['Executive']
 ['from']]
  batch 000: loss=5.9572
  batch 010: loss=5.6866
  batch 020: loss=5.7878
  batch 030: loss=6.1125
  batch 040: loss=6.3008
  batch 050: loss=6.8298
  batch 060: loss=6.2424
  batch 070: loss=5.4658
  batch 080: loss=5.8662
  batch 090: loss=6.0377
  batch 100: loss=6.8240
  batch 110: loss=5.1171
  batch 120: loss=6.2653
  batch 130: loss=5.7929
  batch 140: loss=7.1363

Epoch 3/100
Sample batch: [['the' 'featured']
 ['When' 'the']] [['speaker

In [40]:

# -----------------------------
# prediction

def predict(context_words):
    context_words = np.array(context_words)   # add batch dimension
    y_pred, _ = forwardpropogation(
        windows_to_concat_onehot(context_words, word2id, len(word2id))
    )
    probs = y_pred[-1]
    max_idx = np.argmax(probs)
    
    return id2word[max_idx]    


for x,y in zip(X_test,Y_test):
    print("Context:", x)
    print("True word:", y)
    print("Predicted:", predict(x[:]))


Context: ['that' 'Vandiver']
True word: ['has']


TypeError: 'tuple' object cannot be interpreted as an integer